# Parse Combat Registry Fighter Data Dump

In [ ]:
import gzip

import orjson
import pandas as pd
from tqdm import tqdm

## Constants

In [ ]:
input_path = "../data/raw/combatreg/fighters.jsonl.gz"

## Fighter Metadata

In [ ]:
fighters = []
skip_fields = {
    "public_image_url",
    "current_record",
    "pro_records",
    "am_records",
}

with gzip.open(input_path, "rb") as f:
    for line in f:
        fighter = orjson.loads(line)
        info = fighter["info"]

        fighters.append({
            k: v
            for k, v in info.items()
            if k not in skip_fields
        })

fighters_df = pd.DataFrame(fighters)
fighters_df = fighters_df.drop_duplicates(
    subset="uuid",
    keep="first"
)
fighters_df.to_csv(
    "../data/raw/combatreg/fighters.csv", 
    index=False
)

## Bouts

In [ ]:
bouts = []

with gzip.open(input_path, "rb") as f:
    for line in tqdm(f, desc="Parsing fighters"):
        fighter = orjson.loads(line)

        for history in fighter["fights_history"]:
            try:
                event_section = history["event_section"]
                fight = history["fight"]
                fighters = history["fighters"]
                status = history["status"]

                # Preserve the order CombatReg gives us
                fighter_1 = fighters[0]
                fighter_2 = fighters[1] if len(fighters) > 1 else None

                bouts.append({
                    "uuid": fight["uuid"],
                    "fighter_1_uuid": fighter_1["uuid"],
                    "fighter_2_uuid": fighter_2["uuid"] if fighter_2 else None,
                    "fighter_1_corner": fighter_1["corner"],
                    "fighter_2_corner": fighter_2["corner"] if fighter_2 else None,
                    "round_schedule": fight["round_schedule"],
                    "total_rounds": fight["total_rounds"],
                    "professional": fight["professional"],
                    "exhibition": fight["exhibition"],
                    "title": fight["title"],
                    "section_order": fight["section_order"],
                    "event_name": fight["event_name"],
                    "last_modified_at": fight["last_modified_at"],
                    "event_date": fight["event_date"],
                    "event_city": fight["event_city"],
                    "event_seo_name": fight["event_seo_name"],
                    "event_uuid": fight["event_uuid"],
                    "event_state": fight["event_state"],
                    "event_country": fight["event_country"],
                    "venue": fight["venue"],
                    "official": fight["official"],
                    "live_stat_id": fight["live_stat_id"],
                    "final_stat_id": fight["final_stat_id"],
                    "sport_name": fight["sport_name"],
                    "organization_name": fight["organization_name"],
                    "promotion_name": fight["promotion_name"],
                    "promotion_logo": fight["promotion_logo"],
                    "sanctioning_body_name": fight["sanctioning_body_name"],
                    "referee": fight["referee"],
                    "event_section_type": event_section["event_section_type"],
                    "event_section_starts_at": event_section["starts_at"],
                    "event_section_watch_link": event_section["watch_link"],
                    "event_section_broadcast_partner": event_section["broadcast_partner"],
                    "event_section_name": event_section["name"],
                    "event_section_order": event_section["order"],
                    "status_length": status["length"],
                    "status_result_detail": status["result_detail"],
                    "status_final_round": status["final_round"],
                    "status_final_round_length": status["final_round_length"],
                    "status_weight_division": status["weight_division"],
                    "status_weight_division_abbr": status["weight_division_abbr"],
                    "status_weight_division_bracket_min": status["weight_division_bracket"]["min"] if status["weight_division_bracket"] else None,
                    "status_weight_division_bracket_max": status["weight_division_bracket"]["max"] if status["weight_division_bracket"] else None,
                    "status_result_type": status["result_type"],
                    "winner_uuid": status["winner"]["uuid"] if status["winner"] else None,
                    "winner_corner": status["winner_corner"],
                })

            except Exception as e:
                print("\nERROR parsing fight history:")
                print(f"Error: {type(e).__name__}: {e}")
                print("\nProblematic fight history object:")
                print(history)
                raise

bouts_df = pd.DataFrame(bouts)
bouts_df = bouts_df.drop_duplicates(
    subset="uuid",
    keep="first"
)
bouts_df.to_parquet(
    "../data/raw/combatreg/bouts.parquet",
    index=False,
    compression="zstd"
)